# Data Governance – Credit Card Fraud Detection

This notebook documents and checks the data used by the fraud detection model built in the `credit_card_fraud_detection` notebook. It runs in **Python only** (pandas): no SQL tables and no Unity Catalog needed.

**Dataset:** [Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) on Kaggle (`creditcard.csv`)

## Governance Steps:

### 1. Setup & Data Loading
- Loads the same dataset used by the model

### 2. Dataset Overview
- Source, owner, size, period covered and privacy classification of the dataset

### 3. Feature Register
- One row per column (`Time`, `V1`–`V28`, `Amount`, `Class`) plus the 2 model outputs
- Business meaning, ownership, sensitivity, GDPR flag, critical data element, quality rule
- **Live profile** of each column computed on the data: type, nulls, min, max

### 4. Data Quality Checks
- 8 checks on completeness, validity, consistency, uniqueness and volume
- Result **PASS / FAIL** for each check; *Critical* failures block model training

### 5. Train/Test Split Checks
- Rebuilds the same split as the model notebook (`random_state=42`)
- Verifies size, stratification and no overlap between training and test data

### 6. Model Card
- Data, method, results, intended use, limitations and approval status of the model
- Consistency check of the reported metrics against the confusion matrix

### 7. Governance Summary
- Critical data elements, check results and open items

---
## 1. Setup & Data Loading

In [0]:
%pip install --upgrade kagglehub

In [0]:
# Restart Python on Databricks (skipped on Jupyter / Colab)
try:
    dbutils.library.restartPython()
except NameError:
    pass

In [0]:
import kagglehub
import pandas as pd

pd.set_option("display.max_colwidth", 80)

path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
df = pd.read_csv(f"{path}/creditcard.csv")
print("Rows, columns:", df.shape)

---
## 2. Dataset Overview

General information about the dataset. Size, period and fraud share are computed from the data.

In [0]:
dataset_overview = pd.Series({
    "Dataset name": "Credit Card Fraud Detection (creditcard.csv)",
    "Source": "Kaggle – mlg-ulb/creditcardfraud",
    "Original provider": "Worldline and the Machine Learning Group of ULB (Université Libre de Bruxelles)",
    "Content": "Card transactions made by European cardholders",
    "Rows": f"{len(df):,}",
    "Columns": df.shape[1],
    "Period covered (hours)": round(df["Time"].max() / 3600, 1),
    "Fraud transactions": f"{int(df['Class'].sum()):,} ({df['Class'].mean() * 100:.3f}%)",
    "Personal identifiers": "None: no card number, name or customer ID",
    "Anonymisation": "V1–V28 are PCA components published instead of the original attributes",
    "Overall sensitivity": "Confidential",
    "Used by": "credit_card_fraud_detection notebook (model M001)",
    "Data owner": "Head of Fraud Risk",
    "Data steward": "Fraud Analytics Data Steward",
}, name="Value")
dataset_overview.to_frame()

---
## 3. Feature Register

One row per feature: business meaning, ownership, privacy classification and the data quality rule that applies to it.

In [0]:
OWNER = "Head of Fraud Risk"
STEWARD = "Fraud Analytics Data Steward"

features = []

features.append({
    "feature_id": "F001", "feature_name": "Time", "business_name": "Seconds since first transaction",
    "description": "Seconds between this transaction and the first transaction in the dataset",
    "data_type": "float", "feature_category": "Raw", "allowed_values": "0 to 172800 (two days)",
    "sensitivity": "Internal", "personal_data": "No", "critical_data_element": "No",
    "data_quality_rule": "Never null; between 0 and 172800", "dq_check_ids": "DQ001, DQ005",
    "status": "Active", "known_issues": "Relative time only: no calendar date",
})

for i in range(1, 29):
    features.append({
        "feature_id": f"F{i + 1:03d}", "feature_name": f"V{i}", "business_name": f"Anonymised component {i}",
        "description": f"PCA component {i} of the original transaction attributes, anonymised by the provider",
        "data_type": "float", "feature_category": "Anonymised", "allowed_values": "Any real number, centred on 0",
        "sensitivity": "Confidential", "personal_data": "No (anonymised)", "critical_data_element": "No",
        "data_quality_rule": "Never null", "dq_check_ids": "DQ002",
        "status": "Active", "known_issues": "Meaning unknown: limits explainability",
    })

features.append({
    "feature_id": "F030", "feature_name": "Amount", "business_name": "Transaction amount",
    "description": "Amount of the card transaction (EUR)",
    "data_type": "float", "feature_category": "Raw", "allowed_values": ">= 0",
    "sensitivity": "Confidential", "personal_data": "No", "critical_data_element": "Yes",
    "data_quality_rule": "Never null; never negative", "dq_check_ids": "DQ001, DQ004",
    "status": "Active", "known_issues": None,
})

features.append({
    "feature_id": "F031", "feature_name": "Class", "business_name": "Fraud label (target)",
    "description": "Outcome of the transaction: 1 = fraud, 0 = normal",
    "data_type": "int", "feature_category": "Target", "allowed_values": "0, 1",
    "sensitivity": "Restricted", "personal_data": "No", "critical_data_element": "Yes",
    "data_quality_rule": "Only 0 or 1; fraud share between 0.1% and 0.3%", "dq_check_ids": "DQ001, DQ003, DQ006",
    "status": "Active", "known_issues": "Extremely imbalanced target",
})

features.append({
    "feature_id": "F032", "feature_name": "fraud_probability", "business_name": "Fraud score (model output)",
    "description": "Probability of fraud given by model M001 (predict_proba)",
    "data_type": "float", "feature_category": "Model output", "allowed_values": "0 to 1",
    "sensitivity": "Restricted", "personal_data": "No", "critical_data_element": "Yes",
    "data_quality_rule": "Always between 0 and 1", "dq_check_ids": None,
    "status": "Under review", "known_issues": "Not stored: exists only inside the model notebook",
})

features.append({
    "feature_id": "F033", "feature_name": "fraud_prediction", "business_name": "Fraud alert flag (model output)",
    "description": "1 if fraud_probability >= 0.5, otherwise 0",
    "data_type": "int", "feature_category": "Model output", "allowed_values": "0, 1",
    "sensitivity": "Restricted", "personal_data": "No", "critical_data_element": "Yes",
    "data_quality_rule": "Only 0 or 1", "dq_check_ids": None,
    "status": "Under review", "known_issues": "Many false alarms (low precision)",
})

feature_register = pd.DataFrame(features)
feature_register["source"] = feature_register["feature_category"].map(
    lambda c: "Model M001" if c == "Model output" else "creditcard.csv")
feature_register["data_owner"] = OWNER
feature_register["data_steward"] = STEWARD
feature_register["approved_by"] = None

print("Features registered:", len(feature_register))
feature_register[["feature_id", "feature_name", "business_name", "feature_category",
                  "sensitivity", "critical_data_element", "dq_check_ids", "status"]]

### Live profile of the dataset columns

Compares what the register says with what is actually in the data.

In [0]:
profile = pd.DataFrame({
    "feature_name": df.columns,
    "actual_dtype": df.dtypes.astype(str).values,
    "null_count": df.isnull().sum().values,
    "min": df.min().round(2).values,
    "max": df.max().round(2).values,
    "distinct_values": df.nunique().values,
})

columns_in_register = set(feature_register["feature_name"])
profile["in_register"] = profile["feature_name"].isin(columns_in_register)

feature_register = feature_register.merge(profile, on="feature_name", how="left")

missing = profile.loc[~profile["in_register"], "feature_name"].tolist()
print("Dataset columns documented in the register:", profile["in_register"].sum(), "of", len(profile))
print("Columns missing from the register:", missing if missing else "none")
profile

---
## 4. Data Quality Checks

Each check counts the **rows that break the rule**. A check passes when that number is 0.
- **Critical**: must pass before the data is used to train the model.
- **Warning**: does not block training, but must be reviewed and written in the register.

In [0]:
v_cols = [f"V{i}" for i in range(1, 29)]

dq_checks = [
    ("DQ001", "No nulls in Time, Amount, Class", "Completeness", "Critical",
     lambda d: int(d[["Time", "Amount", "Class"]].isnull().any(axis=1).sum())),
    ("DQ002", "No nulls in V1-V28", "Completeness", "Critical",
     lambda d: int(d[v_cols].isnull().any(axis=1).sum())),
    ("DQ003", "Class is only 0 or 1", "Validity", "Critical",
     lambda d: int((~d["Class"].isin([0, 1])).sum())),
    ("DQ004", "Amount is not negative", "Validity", "Critical",
     lambda d: int((d["Amount"] < 0).sum())),
    ("DQ005", "Time between 0 and 172800 s", "Validity", "Warning",
     lambda d: int(((d["Time"] < 0) | (d["Time"] > 172800)).sum())),
    ("DQ006", "Fraud share between 0.1% and 0.3%", "Consistency", "Critical",
     lambda d: 0 if 0.001 <= d["Class"].mean() <= 0.003 else 1),
    ("DQ007", "No duplicate transactions", "Uniqueness", "Warning",
     lambda d: int(d.duplicated().sum())),
    ("DQ008", "Full extract: 284,807 rows", "Volume", "Critical",
     lambda d: abs(len(d) - 284807)),
]

dq_results = pd.DataFrame(
    [(cid, name, dim, sev, check(df)) for cid, name, dim, sev, check in dq_checks],
    columns=["dq_check_id", "check_name", "dimension", "severity", "failed_rows"])
dq_results["result"] = dq_results["failed_rows"].map(lambda n: "PASS" if n == 0 else "FAIL")
dq_results["run_time"] = pd.Timestamp.now().floor("s")
dq_results

In [0]:
critical_failures = dq_results[(dq_results["result"] == "FAIL") & (dq_results["severity"] == "Critical")]
warnings = dq_results[(dq_results["result"] == "FAIL") & (dq_results["severity"] == "Warning")]

print("Checks passed:   ", (dq_results["result"] == "PASS").sum(), "of", len(dq_results))
print("Critical failures:", len(critical_failures))
print("Warnings:         ", len(warnings))
print()
print("Data APPROVED for model training" if critical_failures.empty
      else "Model training BLOCKED: fix the critical failures first")

### Duplicate transactions

If DQ007 fails, this shows how many duplicates there are and how many of them are frauds, and records the issue in the feature register.

In [0]:
dup_rows = df[df.duplicated(keep=False)]
n_extra = int(df.duplicated().sum())

print("Rows involved in duplicates:", len(dup_rows))
print("Extra copies (rows that could be removed):", n_extra)
print("Fraud rows among the extra copies:", int(df[df.duplicated()]["Class"].sum()))

if n_extra > 0:
    note = f"{n_extra} duplicate rows in the source data (DQ007); kept in training, to be reviewed"
    feature_register.loc[feature_register["source"] == "creditcard.csv", "known_issues"] = \
        feature_register.loc[feature_register["source"] == "creditcard.csv", "known_issues"] \
        .fillna("").map(lambda s: f"{s}; {note}" if s else note)

---
## 5. Train/Test Split Checks

Rebuilds the split used in the model notebook (same settings, `random_state=42`) to verify that it was done correctly. No model is trained here.

In [0]:
from sklearn.model_selection import train_test_split

X = df.drop("Class", axis=1)
y = df["Class"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

split_checks = pd.DataFrame([
    ("Test set is 20% of the data", f"{len(X_test) / len(df):.1%}",
     abs(len(X_test) / len(df) - 0.2) < 0.001),
    ("Fraud share in train matches full data", f"{y_train.mean():.3%} vs {y.mean():.3%}",
     abs(y_train.mean() - y.mean()) < 0.0001),
    ("Fraud share in test matches full data", f"{y_test.mean():.3%} vs {y.mean():.3%}",
     abs(y_test.mean() - y.mean()) < 0.0001),
    ("No row in both train and test", f"{len(set(X_train.index) & set(X_test.index))} shared rows",
     len(set(X_train.index) & set(X_test.index)) == 0),
    ("Target not among the inputs", f"Class in inputs: {'Class' in X.columns}",
     "Class" not in X.columns),
], columns=["check", "observed", "passed"])
split_checks["result"] = split_checks["passed"].map({True: "PASS", False: "FAIL"})

print("Train:", X_train.shape, "frauds:", int(y_train.sum()))
print("Test: ", X_test.shape, "frauds:", int(y_test.sum()))
split_checks.drop(columns="passed")

Exact duplicates can end up one copy in the training set and one in the test set. This makes the test slightly easier than truly new data:

In [0]:
train_rows = pd.concat([X_train, y_train], axis=1)
test_rows = pd.concat([X_test, y_test], axis=1)
leaked = test_rows.merge(train_rows.drop_duplicates(), how="inner", on=list(test_rows.columns))
print("Test rows that also appear identically in the training set:", len(leaked))
print("Of which frauds:", int(leaked["Class"].sum()))

---
## 6. Model Card

Results come from the evaluation on the test set in the `credit_card_fraud_detection` notebook (run on Databricks).

In [0]:
results = {"tp": 90, "fn": 8, "fp": 1467, "tn": 55397,
           "accuracy": 0.9741, "precision": 0.0578, "recall": 0.9184, "f1": 0.1088, "roc_auc": 0.9708}

model_card = pd.Series({
    "Model ID": "M001",
    "Model name": "Credit card fraud classifier",
    "Version": "1.0",
    "Purpose": "Flag card transactions that are likely to be fraud",
    "Algorithm": "Logistic Regression (scikit-learn), max_iter=1000, other settings default",
    "Target": "F031 Class",
    "Inputs": "F001–F030 (Time, V1–V28, Amount), scaled with StandardScaler fitted on training data",
    "Outputs": "F032 fraud_probability, F033 fraud_prediction (threshold 0.5)",
    "Training data": "80% of creditcard.csv: 227,845 rows, 394 frauds",
    "Class balancing": "SMOTE on the training set only: 227,451 normal / 227,451 fraud",
    "Test data": "20% of creditcard.csv, untouched: 56,962 rows, 98 frauds",
    "Frauds caught (TP)": results["tp"],
    "Frauds missed (FN)": results["fn"],
    "False alarms (FP)": results["fp"],
    "Correct normals (TN)": results["tn"],
    "Accuracy": results["accuracy"],
    "Precision (fraud)": results["precision"],
    "Recall (fraud)": results["recall"],
    "F1 (fraud)": results["f1"],
    "ROC AUC": results["roc_auc"],
    "Intended use": "First filter: flagged transactions go to a fraud analyst for review",
    "Not to be used for": "Blocking cards or accounts automatically; data from other periods without revalidation",
    "Limitations": "About 94% of alerts are false alarms; trained on two days of 2013 data; inputs anonymised",
    "Regulatory notes": "Decisions with significant effects on customers need human review (GDPR Art. 22)",
    "Monitoring": "Recall, precision and alert volume on every new data load",
    "Model owner": OWNER,
    "Developer": "alessiokondiak8",
    "Status": "Under review",
    "Approved by": None,
}, name="Value")
model_card.to_frame()

### Consistency check of the reported metrics

Recomputes the metrics from the confusion matrix numbers and compares them with the reported values.

In [0]:
tp, fn, fp, tn = results["tp"], results["fn"], results["fp"], results["tn"]
recomputed = {
    "accuracy": (tp + tn) / (tp + tn + fp + fn),
    "precision": tp / (tp + fp),
    "recall": tp / (tp + fn),
    "f1": 2 * tp / (2 * tp + fp + fn),
}
metric_check = pd.DataFrame([
    (m, results[m], round(v, 4), "PASS" if abs(results[m] - v) < 0.0001 else "FAIL")
    for m, v in recomputed.items()
], columns=["metric", "reported", "recomputed", "result"])

print("Test rows in the confusion matrix:", tp + fn + fp + tn, "(expected", len(X_test), ")")
print("Frauds in the confusion matrix:  ", tp + fn, "(expected", int(y_test.sum()), ")")
metric_check

---
## 7. Governance Summary

### Critical data elements

In [0]:
feature_register.loc[feature_register["critical_data_element"] == "Yes",
                     ["feature_id", "feature_name", "business_name", "sensitivity",
                      "data_quality_rule", "data_owner"]]

### Features by category and sensitivity

In [0]:
pd.crosstab(feature_register["feature_category"], feature_register["sensitivity"])

### Overall status

In [0]:
summary = pd.Series({
    "Features registered": len(feature_register),
    "Dataset columns documented": f"{int(profile['in_register'].sum())} of {len(profile)}",
    "Critical data elements": int((feature_register["critical_data_element"] == "Yes").sum()),
    "Data quality checks passed": f"{(dq_results['result'] == 'PASS').sum()} of {len(dq_results)}",
    "Critical DQ failures": len(critical_failures),
    "DQ warnings": len(warnings),
    "Split checks passed": f"{(split_checks['result'] == 'PASS').sum()} of {len(split_checks)}",
    "Metric consistency checks passed": f"{(metric_check['result'] == 'PASS').sum()} of {len(metric_check)}",
    "Data approved for training": "Yes" if critical_failures.empty else "No",
    "Model status": model_card["Status"],
    "Items waiting for approval": int(feature_register["approved_by"].isnull().sum()) + int(model_card["Approved by"] is None),
}, name="Value")
summary.to_frame()

---
## Conclusions

_Write 3–5 sentences in your own words after running the notebook, for example:_

- Are all 31 dataset columns documented, and which are the critical data elements?
- Did all critical data quality checks pass? Was any warning raised (e.g. duplicates)?
- Was the train/test split correct, and do the reported metrics match the confusion matrix?
- What is still missing before the model can be approved (owner sign-off, open issues)?